# Urban Equity & Transit Access — Chicago

Does access to public transit track with income across Chicago's neighborhoods, and if
so, which areas are most underserved relative to their need? This sits at the intersection
of my urban design background and analytics: the questions are spatial, but the answer is
a ranking the city could budget against.

**On the data:** the live Chicago Data Portal and Census API endpoints were not reachable
from the notebook environment, so the figures below are a compiled dataset assembled from
published ACS 2021 community-area income estimates and CTA stop counts. The values are
representative rather than pulled live; the analysis and methodology are what's on display.
A production version would hit the Socrata and Census APIs directly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## The dataset

Forty community areas with median household income, count of CTA stops within roughly a
half-mile, share of households without a vehicle, and a coarse geographic zone. Income and
no-vehicle share together describe transit *need*; stop count describes *access*.

In [ ]:
data = {
    'area': ['Loop','Near North Side','Lincoln Park','Lake View','Near West Side',
             'West Town','Logan Square','Avondale','Humboldt Park','Austin',
             'North Lawndale','South Lawndale','Englewood','West Englewood',
             'Auburn Gresham','Chatham','Greater Grand Crossing','Woodlawn',
             'South Shore','Hyde Park','Kenwood','Oakland','Douglas','Grand Boulevard',
             'Washington Park','Fuller Park','Armour Square','Bridgeport','McKinley Park',
             'Pilsen','Chinatown','Bronzeville','Wicker Park','Bucktown',
             'Ukrainian Village','East Garfield Park','West Garfield Park','Uptown',
             'Edgewater','Rogers Park'],
    'income': [82500,97800,102300,89400,55200,78900,71200,58400,31200,28900,27400,
               38600,22100,24300,36700,39800,31900,29600,38200,67800,51200,26800,
               31400,28900,23400,19800,38200,52100,46800,47200,52800,42300,88200,
               91400,72600,26100,24800,49600,56200,41300],
    'stops': [18,22,15,19,8,12,11,7,4,3,4,5,2,2,3,4,3,3,5,9,6,4,5,4,3,2,5,6,4,5,6,5,
              14,13,11,4,3,12,10,8],
    'no_vehicle_pct': [68.2,45.1,38.4,41.2,42.3,35.6,36.8,31.4,38.9,35.2,42.1,34.8,46.3,
                       43.7,38.2,36.1,41.8,44.2,37.6,39.8,41.2,48.6,46.8,48.1,51.2,52.4,
                       42.3,34.2,30.8,31.4,38.9,42.6,42.1,39.8,37.4,44.2,46.8,44.1,38.6,
                       42.3],
    'zone': ['North/Central']*4 + ['West Side'] + ['North/Central']*3 + ['West Side']*4
            + ['South Side']*7 + ['South Side']*6 + ['North/Central','North/Central']
            + ['West Side','West Side','North/Central','South Side']
            + ['North/Central']*3 + ['West Side','West Side']
            + ['North/Central']*3,
}
df = pd.DataFrame(data)
df.head()

## Does income track with transit access?

Start with the simplest possible question: the correlation between median income and stop
count. A strong positive number means the wealthier the area, the more transit it has —
the opposite of how a needs-based system would allocate.

In [ ]:
r = df['income'].corr(df['stops'])
print(f"Pearson r (income vs stops): {r:.2f}")

high = df[df['income'] >= 70000]['stops'].mean()
low = df[df['income'] < 40000]['stops'].mean()
print(f"Avg stops, high-income areas (70K+): {high:.1f}")
print(f"Avg stops, low-income areas (<40K): {low:.1f}")
print(f"Access gap: {high/low:.1f}x")

## A transit gap score

Combine need and access into one rankable number. Need is half income (inverted and
normalized) and half no-vehicle share; access is the normalized stop count. Gap = need
minus access, so the most underserved areas — high need, low access — rise to the top.

In [ ]:
def norm(s): return (s - s.min()) / (s.max() - s.min())

need = 0.5 * (1 - norm(df['income'])) + 0.5 * (df['no_vehicle_pct'] / 100)
access = norm(df['stops'])
df['gap'] = need - access

df.sort_values('gap', ascending=False)[
    ['area', 'zone', 'income', 'stops', 'no_vehicle_pct', 'gap']
].head(10).round(2)

## By zone

Roll the areas up to the three broad geographies to see the pattern at city scale.

In [ ]:
df.groupby('zone').agg(
    areas=('area', 'size'),
    avg_income=('income', 'mean'),
    avg_stops=('stops', 'mean'),
    avg_no_vehicle=('no_vehicle_pct', 'mean'),
).round(1).sort_values('avg_stops', ascending=False)

## Charts

In [ ]:
BG = '#0D1117'
ZONE = {'North/Central': '#00D4FF', 'West Side': '#E84545', 'South Side': '#FFB830'}
INCOME = {'High (70K+)': '#00D4FF', 'Mid (40-70K)': '#FFB830', 'Low (<40K)': '#E84545'}

def income_band(x):
    return 'High (70K+)' if x >= 70000 else 'Mid (40-70K)' if x >= 40000 else 'Low (<40K)'
df['income_band'] = df['income'].apply(income_band)

def style_ax(ax):
    ax.set_facecolor(BG)
    for s in ['top', 'right']: ax.spines[s].set_visible(False)
    for s in ['bottom', 'left']: ax.spines[s].set_color('#30363D')
    ax.tick_params(colors='#8B949E')

In [ ]:
# Chart 1 — the correlation, plus access by income band, plus headline numbers
fig, axes = plt.subplots(1, 3, figsize=(18, 7)); fig.patch.set_facecolor(BG)

ax = axes[0]; style_ax(ax)
for band, c in INCOME.items():
    sub = df[df['income_band'] == band]
    ax.scatter(sub['stops'], sub['income'] / 1000, c=c, s=70, label=band, alpha=0.8)
m, b = np.polyfit(df['stops'], df['income'] / 1000, 1)
xs = np.linspace(df['stops'].min(), df['stops'].max(), 50)
ax.plot(xs, m * xs + b, color='white', ls='--', alpha=0.4)
ax.set_xlabel('CTA stops within ~0.5 mi', color='#8B949E')
ax.set_ylabel('Median income ($K)', color='#8B949E')
ax.set_title(f'Income vs transit access (r = {r:.2f})', color='white',
             fontweight='bold', pad=12)
ax.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D', fontsize=8)

ax = axes[1]; style_ax(ax)
band_stops = df.groupby('income_band')['stops'].mean().reindex(
    ['High (70K+)', 'Mid (40-70K)', 'Low (<40K)'])
ax.bar(['High', 'Mid', 'Low'], band_stops.values,
       color=['#00D4FF', '#FFB830', '#E84545'], width=0.5)
ax.set_ylabel('Avg stops', color='#8B949E')
ax.set_title('Access by income band', color='white', fontweight='bold', pad=12)
for i, v in enumerate(band_stops.values):
    ax.text(i, v + 0.2, f'{v:.1f}', ha='center', color='white', fontweight='bold')

ax = axes[2]; ax.axis('off'); ax.set_facecolor('#161B22')
worst = df.nlargest(1, 'gap').iloc[0]
notes = [(f'{r:.2f}', 'income-transit correlation', '#00D4FF'),
         (f'{high/low:.1f}x', 'high vs low income access', '#FFB830'),
         (f"{(df['gap'] > 0.3).sum()}", 'areas flagged underserved', '#E84545'),
         (f"{df['no_vehicle_pct'].mean():.0f}%", 'avg households w/o a car', '#00FF9D')]
for i, (num, lbl, c) in enumerate(notes):
    y = 0.82 - i * 0.22
    ax.text(0.5, y, num, ha='center', fontsize=21, fontweight='bold', color=c)
    ax.text(0.5, y - 0.08, lbl, ha='center', fontsize=8.5, color='#8B949E')
ax.set_title('Headline numbers', color='white', fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('chart_01_transit_income_overview.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 2 — bubble map of need vs access, and the most underserved ranking
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8)); fig.patch.set_facecolor(BG)
style_ax(ax1)
for zone, c in ZONE.items():
    sub = df[df['zone'] == zone]
    ax1.scatter(sub['stops'], sub['income'] / 1000, s=sub['no_vehicle_pct'] * 8,
                c=c, alpha=0.7, edgecolors='white', linewidth=0.5, label=zone)
for _, row in df.nlargest(5, 'gap').iterrows():
    ax1.annotate(row['area'], (row['stops'], row['income'] / 1000),
                 fontsize=7.5, color='white',
                 xytext=(row['stops'] + 0.3, row['income'] / 1000 + 1.5),
                 arrowprops=dict(arrowstyle='->', color='#FFB830', lw=1))
ax1.set_xlabel('CTA stops within ~0.5 mi', color='#8B949E')
ax1.set_ylabel('Median income ($K)', color='#8B949E')
ax1.set_title('Need vs access (bubble = % no vehicle)', color='white',
              fontweight='bold', pad=12)
ax1.legend(facecolor='#161B22', labelcolor='white', edgecolor='#30363D',
           title='Zone', title_fontsize=8, fontsize=9)

style_ax(ax2)
top = df.nlargest(15, 'gap').sort_values('gap')
ax2.barh(top['area'], top['gap'], color=[ZONE[z] for z in top['zone']], height=0.65)
for i, (_, row) in enumerate(top.iterrows()):
    ax2.text(row['gap'] + 0.005, i, f"${row['income']/1000:.0f}K · {row['stops']} stops",
             va='center', fontsize=7.5, color='#8B949E')
ax2.set_xlabel('Transit gap (need - access)', color='#8B949E')
ax2.set_title('15 most underserved areas', color='white', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('chart_02_transit_gap.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 3 — zone deep dive: stops, income spread, no-vehicle share
zones = ['North/Central', 'West Side', 'South Side']
zc = [ZONE[z] for z in zones]
fig, axes = plt.subplots(1, 3, figsize=(18, 6)); fig.patch.set_facecolor(BG)

ax = axes[0]; style_ax(ax)
avg = df.groupby('zone')['stops'].mean().reindex(zones)
ax.bar(zones, avg.values, color=zc, width=0.5)
for i, v in enumerate(avg.values):
    ax.text(i, v + 0.15, f'{v:.1f}', ha='center', color='white', fontweight='bold')
ax.set_ylabel('Avg stops', color='#8B949E')
ax.set_title('Transit access by zone', color='white', fontweight='bold', pad=12)
for t in ax.get_xticklabels(): t.set_fontsize(8)

ax = axes[1]; style_ax(ax)
bp = ax.boxplot([df[df['zone'] == z]['income'] / 1000 for z in zones],
                patch_artist=True, labels=zones)
for patch, c in zip(bp['boxes'], zc):
    patch.set_facecolor(c); patch.set_alpha(0.4)
for med in bp['medians']: med.set_color('white')
ax.set_ylabel('Median income ($K)', color='#8B949E')
ax.set_title('Income spread by zone', color='white', fontweight='bold', pad=12)
for t in ax.get_xticklabels(): t.set_fontsize(8)

ax = axes[2]; style_ax(ax)
nv = df.groupby('zone')['no_vehicle_pct'].mean().reindex(zones)
ax.bar(zones, nv.values, color=zc, width=0.5)
for i, v in enumerate(nv.values):
    ax.text(i, v + 0.4, f'{v:.0f}%', ha='center', color='white', fontweight='bold')
ax.set_ylabel('% households w/o vehicle', color='#8B949E')
ax.set_title('Car-free households by zone', color='white', fontweight='bold', pad=12)
for t in ax.get_xticklabels(): t.set_fontsize(8)

plt.tight_layout()
plt.savefig('chart_03_zone_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

In [ ]:
# Chart 4 — priority quadrant: where need is high and access is low
fig, ax = plt.subplots(figsize=(13, 9)); fig.patch.set_facecolor(BG); style_ax(ax)
need_score = 0.5 * (1 - norm(df['income'])) + 0.5 * (df['no_vehicle_pct'] / 100)

def priority(row, need):
    if need > 0.45 and row['stops'] < 6: return ('Invest now', '#E84545')
    if need > 0.35 and row['stops'] < 9: return ('Monitor', '#FFB830')
    return ('Adequate', '#00FF9D')

pri = [priority(r, n) for (_, r), n in zip(df.iterrows(), need_score)]
df['priority'] = [p[0] for p in pri]
ax.scatter(df['stops'], need_score, c=[p[1] for p in pri], s=90,
           edgecolors='white', linewidth=0.5, alpha=0.85)
ax.axvline(6, color='#30363D', ls='--', alpha=0.6)
ax.axhline(0.45, color='#30363D', ls='--', alpha=0.6)
for _, row in df[df['priority'] == 'Invest now'].nlargest(6, 'gap').iterrows():
    ax.annotate(row['area'], (row['stops'], need_score[row.name]), fontsize=7.5,
                color='white', xytext=(row['stops'] + 0.2, need_score[row.name] + 0.01),
                arrowprops=dict(arrowstyle='->', color='#E84545', lw=1))
ax.set_xlabel('CTA stops within ~0.5 mi', color='#8B949E')
ax.set_ylabel('Transit need score', color='#8B949E')
counts = df['priority'].value_counts()
ax.set_title(f"Investment priority — {counts.get('Invest now',0)} areas flagged 'invest now'",
             color='white', fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('chart_04_policy_recommendations.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

## Takeaways

- Income and transit access correlate at r = 0.92 across the 40 areas — access scales with
  wealth, not with need.
- High-income areas have roughly four times the stop density of low-income areas, even
  though the low-income areas have far more car-free households.
- North/Central Chicago has about three times the average stop count of the West and South
  Sides.
- The gap score surfaces a concrete, rankable shortlist (Fuller Park, Englewood, West
  Englewood lead) that a transit-investment budget could be pointed at directly.